# 04 — Source Expansion and Retrieval Evaluation

## Responsible AI RAG Assistant

This notebook expands the source base of the Responsible AI RAG Assistant.

The previous notebooks created:

- document ingestion and chunking
- local embeddings
- Chroma vector store
- semantic retrieval
- context-only RAG answer baseline

The current limitation is that the vector store contains only one source:

`SRC-001 — European Commission AI Act overview page`

This notebook strengthens the project by adding more authoritative public sources.

## Notebook Goals

By the end of this notebook, we want to:

1. Load the existing source inventory.
2. Collect additional official/public sources.
3. Extract text from `.txt` and `.pdf` files.
4. Clean and chunk all collected sources.
5. Save combined processed chunks.
6. Rebuild the Chroma vector store with multiple sources.
7. Evaluate retrieval quality using a broader test set.
8. Save retrieval evaluation results.

## Planned Source Expansion

The project will expand from one source to five sources:

1. `SRC-001` — European Commission AI Act overview page
2. `SRC-002` — Official EU AI Act legal text from EUR-Lex
3. `SRC-003` — Official GDPR legal text from EUR-Lex
4. `SRC-004` — NIST AI Risk Management Framework PDF
5. `SRC-005` — NIST AI RMF overview page

## Responsible-Use Note

This project uses public and authoritative sources only.

The assistant is a knowledge-support prototype and does not provide legal advice.

Raw downloaded files, generated vector stores, `.env` files, and private documents should not be uploaded to GitHub.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
from datetime import date

print("Notebook 04 setup started.")
print(f"Current date: {date.today()}")

Notebook 04 setup started.
Current date: 2026-07-02


In [2]:
# Project paths

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
VECTOR_STORE_DIR = DATA_DIR / "vector_store"
DOCS_DIR = PROJECT_ROOT / "docs"
REPORTS_DIR = PROJECT_ROOT / "reports"

paths = {
    "PROJECT_ROOT": PROJECT_ROOT,
    "DATA_DIR": DATA_DIR,
    "RAW_DATA_DIR": RAW_DATA_DIR,
    "PROCESSED_DATA_DIR": PROCESSED_DATA_DIR,
    "VECTOR_STORE_DIR": VECTOR_STORE_DIR,
    "DOCS_DIR": DOCS_DIR,
    "REPORTS_DIR": REPORTS_DIR,
}

for name, path in paths.items():
    print(f"{name}: {path}")
    print(f"Exists: {path.exists()}")
    print("-" * 80)

PROJECT_ROOT: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant
Exists: True
--------------------------------------------------------------------------------
DATA_DIR: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data
Exists: True
--------------------------------------------------------------------------------
RAW_DATA_DIR: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\raw
Exists: True
--------------------------------------------------------------------------------
PROCESSED_DATA_DIR: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\processed
Exists: True
--------------------------------------------------------------------------------
VECTOR_STORE_DIR: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\vector_store
Exists: True
--------------------------------------------------------------------------------
DOCS_DIR: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\docs
Exists: True
-------------------------------------------------------------------

## Load Existing Source Inventory

The source inventory was created in Notebook 01 and updated after processing `SRC-001`.

This notebook loads the existing inventory and checks the current source status before collecting additional documents.

In [3]:
# Load existing source inventory

source_inventory_path = PROCESSED_DATA_DIR / "source_inventory.csv"

if not source_inventory_path.exists():
    raise FileNotFoundError(f"Source inventory not found: {source_inventory_path}")

source_inventory_df = pd.read_csv(source_inventory_path)

print(f"Loaded source inventory: {source_inventory_path}")
print(f"Number of sources: {len(source_inventory_df)}")
print(f"Columns: {list(source_inventory_df.columns)}")

source_inventory_df[
    [
        "source_id",
        "source_name",
        "publisher",
        "source_type",
        "collection_status",
        "processing_status",
    ]
]

Loaded source inventory: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\processed\source_inventory.csv
Number of sources: 5
Columns: ['source_id', 'source_name', 'publisher', 'source_type', 'main_topic', 'url', 'planned_use', 'planned_raw_filename', 'planned_raw_path', 'collection_status', 'processing_status', 'include_in_version_1', 'raw_file_exists', 'raw_word_count', 'raw_character_count', 'processed_chunks_file', 'chunk_count']


,source_id,source_name,publisher,source_type,collection_status,processing_status
0,SRC-001,AI Act,European Commission,Official web page,collected,chunked
1,SRC-002,Regulation (EU) 2024/1689 Artificial Intellige...,EUR-Lex / European Union,Official legal text,not_collected_yet,not_processed_yet
2,SRC-003,Regulation (EU) 2016/679 General Data Protecti...,EUR-Lex / European Union,Official legal text,not_collected_yet,not_processed_yet
3,SRC-004,Artificial Intelligence Risk Management Framew...,NIST,Public framework PDF,not_collected_yet,not_processed_yet
4,SRC-005,AI Risk Management Framework overview page,NIST,Official web page,not_collected_yet,not_processed_yet


## Collect Additional Raw Sources

The next step is to collect the remaining public and authoritative source files.

Planned collection targets:

1. `SRC-002` — Official EU AI Act legal text from EUR-Lex
2. `SRC-003` — Official GDPR legal text from EUR-Lex
3. `SRC-004` — NIST AI Risk Management Framework PDF
4. `SRC-005` — NIST AI RMF overview page

Raw files will be saved locally in `data/raw/`.

These raw files are local development inputs and should not be uploaded to GitHub.

In [4]:
import requests
from bs4 import BeautifulSoup


def clean_extracted_text(text: str) -> str:
    """
    Clean extracted text by normalizing whitespace.
    """
    if not isinstance(text, str):
        return ""
    
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    
    return text


def fetch_web_page_text(url: str) -> dict:
    """
    Fetch a public web page and extract readable text.

    This function is used for public official web pages and legal text pages.
    """
    headers = {
        "User-Agent": (
            "Mozilla/5.0 "
            "(compatible; ResponsibleAIRAGAssistant/1.0; educational portfolio project)"
        )
    }
    
    response = requests.get(url, headers=headers, timeout=60)
    response.raise_for_status()
    
    soup = BeautifulSoup(response.text, "html.parser")
    
    for tag in soup(["script", "style", "noscript", "svg"]):
        tag.decompose()
    
    title = soup.title.get_text(" ", strip=True) if soup.title else ""
    
    main_content = soup.find("main") or soup.find("article") or soup.body or soup
    
    text_elements = []
    for element in main_content.find_all(["h1", "h2", "h3", "h4", "p", "li", "td"]):
        text = element.get_text(" ", strip=True)
        if text:
            text_elements.append(text)
    
    extracted_text = "\n".join(text_elements)
    cleaned_text = clean_extracted_text(extracted_text)
    
    lower_text = cleaned_text.lower()
    blocked_indicators = [
        "verify that you're not a robot",
        "javascript is disabled",
        "enable javascript",
    ]
    
    possible_blocked_page = any(indicator in lower_text for indicator in blocked_indicators)
    
    return {
        "url": url,
        "title": title,
        "status_code": response.status_code,
        "text": extracted_text,
        "cleaned_text": cleaned_text,
        "character_count": len(extracted_text),
        "word_count": len(extracted_text.split()),
        "possible_blocked_page": possible_blocked_page,
    }


def download_binary_file(url: str, output_path: Path) -> dict:
    """
    Download a public binary file, such as a PDF.
    """
    headers = {
        "User-Agent": (
            "Mozilla/5.0 "
            "(compatible; ResponsibleAIRAGAssistant/1.0; educational portfolio project)"
        )
    }
    
    response = requests.get(url, headers=headers, timeout=120)
    response.raise_for_status()
    
    output_path.write_bytes(response.content)
    
    return {
        "url": url,
        "status_code": response.status_code,
        "file_path": output_path,
        "file_exists": output_path.exists(),
        "file_size_bytes": output_path.stat().st_size if output_path.exists() else 0,
    }


def ensure_inventory_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ensure inventory has the columns needed for collection and processing tracking.
    """
    required_tracking_columns = {
        "raw_file_exists": False,
        "raw_word_count": np.nan,
        "raw_character_count": np.nan,
        "raw_file_size_bytes": np.nan,
        "collection_notes": "",
    }
    
    for column, default_value in required_tracking_columns.items():
        if column not in df.columns:
            df[column] = default_value
    
    return df


source_inventory_df = ensure_inventory_columns(source_inventory_df)

print("Source collection helper functions are ready.")

Source collection helper functions are ready.


In [5]:
# Collect remaining raw sources

collection_targets = ["SRC-002", "SRC-003", "SRC-004", "SRC-005"]

collection_results = []

for source_id in collection_targets:
    row = source_inventory_df.loc[source_inventory_df["source_id"] == source_id].iloc[0]
    
    source_name = row["source_name"]
    source_type = row["source_type"]
    url = row["url"]
    planned_raw_filename = row["planned_raw_filename"]
    output_path = RAW_DATA_DIR / planned_raw_filename
    
    print("=" * 100)
    print(f"Collecting {source_id}: {source_name}")
    print(f"Source type: {source_type}")
    print(f"Output path: {output_path}")
    
    try:
        if planned_raw_filename.lower().endswith(".pdf"):
            result = download_binary_file(url, output_path)
            
            collection_status = "collected"
            processing_status = "raw_pdf_saved"
            raw_word_count = np.nan
            raw_character_count = np.nan
            raw_file_size_bytes = result["file_size_bytes"]
            notes = "PDF downloaded successfully."
            
            print(f"Downloaded PDF file size: {raw_file_size_bytes} bytes")
        
        else:
            result = fetch_web_page_text(url)
            
            if result["possible_blocked_page"]:
                collection_status = "collection_warning"
                processing_status = "possible_blocked_page"
                notes = "Page may require JavaScript or bot verification. Manual fallback may be needed."
            elif result["word_count"] < 100:
                collection_status = "collection_warning"
                processing_status = "low_text_extraction"
                notes = "Extracted text is very short. Manual review needed."
            else:
                collection_status = "collected"
                processing_status = "raw_text_saved"
                notes = "Text extracted successfully."
            
            output_path.write_text(result["text"], encoding="utf-8")
            
            raw_word_count = result["word_count"]
            raw_character_count = result["character_count"]
            raw_file_size_bytes = output_path.stat().st_size if output_path.exists() else 0
            
            print(f"Status code: {result['status_code']}")
            print(f"Title: {result['title']}")
            print(f"Word count: {raw_word_count}")
            print(f"Character count: {raw_character_count}")
            print(f"Possible blocked page: {result['possible_blocked_page']}")
            print("Preview:")
            print(result["cleaned_text"][:700])
        
        raw_file_exists = output_path.exists()
        
    except Exception as error:
        collection_status = "collection_failed"
        processing_status = "not_processed_yet"
        raw_file_exists = output_path.exists()
        raw_word_count = np.nan
        raw_character_count = np.nan
        raw_file_size_bytes = output_path.stat().st_size if output_path.exists() else np.nan
        notes = f"Collection failed: {error}"
        
        print(f"ERROR collecting {source_id}: {error}")
    
    # Update inventory
    source_inventory_df.loc[
        source_inventory_df["source_id"] == source_id,
        "collection_status"
    ] = collection_status
    
    source_inventory_df.loc[
        source_inventory_df["source_id"] == source_id,
        "processing_status"
    ] = processing_status
    
    source_inventory_df.loc[
        source_inventory_df["source_id"] == source_id,
        "raw_file_exists"
    ] = raw_file_exists
    
    source_inventory_df.loc[
        source_inventory_df["source_id"] == source_id,
        "raw_word_count"
    ] = raw_word_count
    
    source_inventory_df.loc[
        source_inventory_df["source_id"] == source_id,
        "raw_character_count"
    ] = raw_character_count
    
    source_inventory_df.loc[
        source_inventory_df["source_id"] == source_id,
        "raw_file_size_bytes"
    ] = raw_file_size_bytes
    
    source_inventory_df.loc[
        source_inventory_df["source_id"] == source_id,
        "collection_notes"
    ] = notes
    
    collection_results.append(
        {
            "source_id": source_id,
            "source_name": source_name,
            "collection_status": collection_status,
            "processing_status": processing_status,
            "raw_file_exists": raw_file_exists,
            "raw_word_count": raw_word_count,
            "raw_character_count": raw_character_count,
            "raw_file_size_bytes": raw_file_size_bytes,
            "collection_notes": notes,
        }
    )

collection_results_df = pd.DataFrame(collection_results)

print("=" * 100)
print("Collection run completed.")

collection_results_df

Source type: Official legal text
Output path: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\raw\src_002_eurlex_ai_act_regulation_2024_1689.txt
Status code: 202
Title: 
Word count: 0
Character count: 0
Possible blocked page: False
Preview:

Source type: Official legal text
Output path: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\raw\src_003_eurlex_gdpr_regulation_2016_679.txt
Status code: 202
Title: 
Word count: 0
Character count: 0
Possible blocked page: False
Preview:

Source type: Public framework PDF
Output path: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\raw\src_004_nist_ai_rmf_1_0.pdf
Downloaded PDF file size: 1946127 bytes
Source type: Official web page
Output path: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\raw\src_005_nist_ai_rmf_overview.txt
Status code: 200
Title: AI Risk Management Framework | NIST
Word count: 11
Character count: 89
Possible blocked page: False
Preview:
NIST Risk Management Framework Aims to Improve Trustwo

,source_id,source_name,collection_status,processing_status,raw_file_exists,raw_word_count,raw_character_count,raw_file_size_bytes,collection_notes
0,SRC-002,Regulation (EU) 2024/1689 Artificial Intellige...,collection_warning,low_text_extraction,True,0.0,0.0,0,Extracted text is very short. Manual review ne...
1,SRC-003,Regulation (EU) 2016/679 General Data Protecti...,collection_warning,low_text_extraction,True,0.0,0.0,0,Extracted text is very short. Manual review ne...
2,SRC-004,Artificial Intelligence Risk Management Framew...,collected,raw_pdf_saved,True,NaN,NaN,1946127,PDF downloaded successfully.
3,SRC-005,AI Risk Management Framework overview page,collection_warning,low_text_extraction,True,11.0,89.0,89,Extracted text is very short. Manual review ne...


## Repair Source Collection

The first collection attempt showed that some official web pages were not extracted correctly.

Observed issues:

- `SRC-002` returned an empty text file.
- `SRC-003` returned an empty text file.
- `SRC-005` returned only a very short text extract.

This is a normal issue when collecting text from legal or government web pages.

Repair strategy:

1. Use official EUR-Lex PDF endpoints for the AI Act and GDPR.
2. Keep the NIST AI RMF PDF already downloaded.
3. Re-extract the NIST overview page using a broader text-extraction method.
4. Update the source inventory with repaired raw filenames and collection status.

Raw files remain local and should not be uploaded to GitHub.

In [6]:
from bs4 import BeautifulSoup
import requests


def download_pdf_with_validation(url: str, output_path: Path) -> dict:
    """
    Download a PDF file and validate that the saved file looks like a real PDF.
    """
    headers = {
        "User-Agent": (
            "Mozilla/5.0 "
            "(compatible; ResponsibleAIRAGAssistant/1.0; educational portfolio project)"
        ),
        "Accept": "application/pdf,text/html,*/*",
    }

    response = requests.get(url, headers=headers, timeout=120, allow_redirects=True)
    response.raise_for_status()

    output_path.write_bytes(response.content)

    file_size = output_path.stat().st_size if output_path.exists() else 0

    is_pdf = (
        output_path.exists()
        and file_size > 50_000
        and output_path.read_bytes()[:4] == b"%PDF"
    )

    return {
        "url": url,
        "status_code": response.status_code,
        "output_path": output_path,
        "file_exists": output_path.exists(),
        "file_size_bytes": file_size,
        "is_valid_pdf": is_pdf,
    }


def fetch_nist_overview_text(url: str) -> dict:
    """
    Fetch the NIST AI RMF overview page with broader extraction logic.
    """
    headers = {
        "User-Agent": (
            "Mozilla/5.0 "
            "(compatible; ResponsibleAIRAGAssistant/1.0; educational portfolio project)"
        )
    }

    response = requests.get(url, headers=headers, timeout=60)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    for tag in soup(["script", "style", "noscript", "svg"]):
        tag.decompose()

    raw_lines = soup.get_text("\n", strip=True).splitlines()

    cleaned_lines = []
    for line in raw_lines:
        line = clean_extracted_text(line)

        if not line:
            continue

        # Remove very short navigation fragments
        if len(line.split()) < 3:
            continue

        # Remove common navigation/footer fragments
        lower_line = line.lower()
        skip_terms = [
            "skip to main content",
            "search nist",
            "menu",
            "close",
            "was this page helpful",
            "webmaster",
            "headquarters",
            "facebook",
            "linkedin",
            "instagram",
            "youtube",
        ]

        if any(term in lower_line for term in skip_terms):
            continue

        cleaned_lines.append(line)

    # Try to focus on the overview section if it exists
    start_index = 0
    end_index = len(cleaned_lines)

    for i, line in enumerate(cleaned_lines):
        if "Overview of the AI RMF" in line:
            start_index = i
            break

    for i, line in enumerate(cleaned_lines[start_index:], start=start_index):
        if line in ["Translations", "Prior Documents", "Contact Us"]:
            end_index = i
            break

    selected_lines = cleaned_lines[start_index:end_index]

    # Fallback: if section extraction is too short, use broader cleaned content
    selected_text = "\n".join(selected_lines)
    if len(selected_text.split()) < 80:
        selected_text = "\n".join(cleaned_lines)

    selected_text = clean_extracted_text(selected_text)

    return {
        "url": url,
        "status_code": response.status_code,
        "text": selected_text,
        "word_count": len(selected_text.split()),
        "character_count": len(selected_text),
        "preview": selected_text[:800],
    }


repair_targets = [
    {
        "source_id": "SRC-002",
        "new_raw_filename": "src_002_eurlex_ai_act_regulation_2024_1689.pdf",
        "url": "https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=OJ:L_202401689",
        "repair_type": "pdf",
    },
    {
        "source_id": "SRC-003",
        "new_raw_filename": "src_003_eurlex_gdpr_regulation_2016_679.pdf",
        "url": "https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=CELEX:32016R0679",
        "repair_type": "pdf",
    },
    {
        "source_id": "SRC-005",
        "new_raw_filename": "src_005_nist_ai_rmf_overview.txt",
        "url": "https://www.nist.gov/itl/ai-risk-management-framework",
        "repair_type": "web_text",
    },
]

repair_results = []

for target in repair_targets:
    source_id = target["source_id"]
    output_path = RAW_DATA_DIR / target["new_raw_filename"]

    print("=" * 100)
    print(f"Repairing {source_id}")
    print(f"Output path: {output_path}")

    try:
        if target["repair_type"] == "pdf":
            result = download_pdf_with_validation(target["url"], output_path)

            if result["is_valid_pdf"]:
                collection_status = "collected"
                processing_status = "raw_pdf_saved"
                notes = "Repaired using official PDF source."
            else:
                collection_status = "collection_warning"
                processing_status = "pdf_validation_failed"
                notes = "Downloaded file does not look like a valid PDF. Manual review needed."

            raw_word_count = np.nan
            raw_character_count = np.nan
            raw_file_size_bytes = result["file_size_bytes"]
            raw_file_exists = result["file_exists"]

            print(f"Status code: {result['status_code']}")
            print(f"File exists: {raw_file_exists}")
            print(f"File size: {raw_file_size_bytes}")
            print(f"Valid PDF: {result['is_valid_pdf']}")

        else:
            result = fetch_nist_overview_text(target["url"])

            output_path.write_text(result["text"], encoding="utf-8")

            raw_word_count = result["word_count"]
            raw_character_count = result["character_count"]
            raw_file_size_bytes = output_path.stat().st_size if output_path.exists() else 0
            raw_file_exists = output_path.exists()

            if raw_word_count >= 80:
                collection_status = "collected"
                processing_status = "raw_text_saved"
                notes = "Repaired using broader text extraction."
            else:
                collection_status = "collection_warning"
                processing_status = "low_text_extraction"
                notes = "Text extraction still short. Manual review needed."

            print(f"Status code: {result['status_code']}")
            print(f"Word count: {raw_word_count}")
            print(f"Character count: {raw_character_count}")
            print("Preview:")
            print(result["preview"])

        # Update source inventory with repaired file details
        source_inventory_df.loc[
            source_inventory_df["source_id"] == source_id,
            "url"
        ] = target["url"]

        source_inventory_df.loc[
            source_inventory_df["source_id"] == source_id,
            "planned_raw_filename"
        ] = target["new_raw_filename"]

        source_inventory_df.loc[
            source_inventory_df["source_id"] == source_id,
            "planned_raw_path"
        ] = str(output_path)

        source_inventory_df.loc[
            source_inventory_df["source_id"] == source_id,
            "collection_status"
        ] = collection_status

        source_inventory_df.loc[
            source_inventory_df["source_id"] == source_id,
            "processing_status"
        ] = processing_status

        source_inventory_df.loc[
            source_inventory_df["source_id"] == source_id,
            "raw_file_exists"
        ] = raw_file_exists

        source_inventory_df.loc[
            source_inventory_df["source_id"] == source_id,
            "raw_word_count"
        ] = raw_word_count

        source_inventory_df.loc[
            source_inventory_df["source_id"] == source_id,
            "raw_character_count"
        ] = raw_character_count

        source_inventory_df.loc[
            source_inventory_df["source_id"] == source_id,
            "raw_file_size_bytes"
        ] = raw_file_size_bytes

        source_inventory_df.loc[
            source_inventory_df["source_id"] == source_id,
            "collection_notes"
        ] = notes

    except Exception as error:
        collection_status = "collection_failed"
        processing_status = "not_processed_yet"
        raw_file_exists = output_path.exists()
        raw_word_count = np.nan
        raw_character_count = np.nan
        raw_file_size_bytes = output_path.stat().st_size if output_path.exists() else np.nan
        notes = f"Repair failed: {error}"

        print(f"ERROR repairing {source_id}: {error}")

    repair_results.append(
        {
            "source_id": source_id,
            "collection_status": collection_status,
            "processing_status": processing_status,
            "raw_file_exists": raw_file_exists,
            "raw_word_count": raw_word_count,
            "raw_character_count": raw_character_count,
            "raw_file_size_bytes": raw_file_size_bytes,
            "collection_notes": notes,
        }
    )

repair_results_df = pd.DataFrame(repair_results)

print("=" * 100)
print("Repair run completed.")

repair_results_df

Repairing SRC-002
Output path: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\raw\src_002_eurlex_ai_act_regulation_2024_1689.pdf
Status code: 202
File exists: True
File size: 2035
Valid PDF: False
Repairing SRC-003
Output path: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\raw\src_003_eurlex_gdpr_regulation_2016_679.pdf
Status code: 202
File exists: True
File size: 2035
Valid PDF: False
Repairing SRC-005
Output path: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\raw\src_005_nist_ai_rmf_overview.txt
Status code: 200
Word count: 562
Character count: 3645
Preview:
Overview of the AI RMF Led by the Information Technology Laboratory (ITL) AI Program, and in collaboration with the private and public sectors, NIST has developed a framework to better manage risks to individuals, organizations, and society associated with artificial intelligence (AI). The NIST AI Risk Management Framework (AI RMF) is intended for voluntary use and to improve the ability to incorp

,source_id,collection_status,processing_status,raw_file_exists,raw_word_count,raw_character_count,raw_file_size_bytes,collection_notes
0,SRC-002,collection_warning,pdf_validation_failed,True,NaN,NaN,2035,Downloaded file does not look like a valid PDF...
1,SRC-003,collection_warning,pdf_validation_failed,True,NaN,NaN,2035,Downloaded file does not look like a valid PDF...
2,SRC-005,collected,raw_text_saved,True,562.0,3645.0,3647,Repaired using broader text extraction.


In [7]:
# Save repaired source inventory

source_inventory_df.to_csv(source_inventory_path, index=False, encoding="utf-8")

print(f"Saved repaired source inventory to: {source_inventory_path}")
print(f"File exists: {source_inventory_path.exists()}")

source_inventory_df[
    [
        "source_id",
        "source_name",
        "planned_raw_filename",
        "collection_status",
        "processing_status",
        "raw_file_exists",
        "raw_word_count",
        "raw_character_count",
        "raw_file_size_bytes",
        "collection_notes",
    ]
]

Saved repaired source inventory to: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\processed\source_inventory.csv
File exists: True


,source_id,source_name,planned_raw_filename,collection_status,processing_status,raw_file_exists,raw_word_count,raw_character_count,raw_file_size_bytes,collection_notes
0,SRC-001,AI Act,src_001_european_commission_ai_act_overview.txt,collected,chunked,True,2229.0,14464.0,NaN,
1,SRC-002,Regulation (EU) 2024/1689 Artificial Intellige...,src_002_eurlex_ai_act_regulation_2024_1689.pdf,collection_warning,pdf_validation_failed,True,NaN,NaN,2035.0,Downloaded file does not look like a valid PDF...
2,SRC-003,Regulation (EU) 2016/679 General Data Protecti...,src_003_eurlex_gdpr_regulation_2016_679.pdf,collection_warning,pdf_validation_failed,True,NaN,NaN,2035.0,Downloaded file does not look like a valid PDF...
3,SRC-004,Artificial Intelligence Risk Management Framew...,src_004_nist_ai_rmf_1_0.pdf,collected,raw_pdf_saved,True,NaN,NaN,1946127.0,PDF downloaded successfully.
4,SRC-005,AI Risk Management Framework overview page,src_005_nist_ai_rmf_overview.txt,collected,raw_text_saved,True,562.0,3645.0,3647.0,Repaired using broader text extraction.


In [8]:
# Check raw source files after repair

raw_file_records = []

for _, row in source_inventory_df.iterrows():
    raw_path = RAW_DATA_DIR / row["planned_raw_filename"]

    raw_file_records.append(
        {
            "source_id": row["source_id"],
            "planned_raw_filename": row["planned_raw_filename"],
            "raw_file_exists": raw_path.exists(),
            "file_size_bytes": raw_path.stat().st_size if raw_path.exists() else None,
        }
    )

raw_files_status_df = pd.DataFrame(raw_file_records)
raw_files_status_df

,source_id,planned_raw_filename,raw_file_exists,file_size_bytes
0,SRC-001,src_001_european_commission_ai_act_overview.txt,True,14612
1,SRC-002,src_002_eurlex_ai_act_regulation_2024_1689.pdf,True,2035
2,SRC-003,src_003_eurlex_gdpr_regulation_2016_679.pdf,True,2035
3,SRC-004,src_004_nist_ai_rmf_1_0.pdf,True,1946127
4,SRC-005,src_005_nist_ai_rmf_overview.txt,True,3647


## Repair EUR-Lex Legal Text Sources

The previous PDF repair attempt did not produce valid PDF files for:

- `SRC-002` — EU AI Act legal text
- `SRC-003` — GDPR legal text

The saved files were too small and failed PDF validation.

Repair strategy:

1. Use the official EUR-Lex legal text pages instead of the PDF endpoint.
2. Extract broad readable page text.
3. Remove obvious navigation and JavaScript-warning fragments where possible.
4. Save the repaired legal text files as `.txt` files in `data/raw/`.
5. Update the source inventory.

These files remain local raw development inputs and should not be uploaded to GitHub.

In [11]:
def fetch_eurlex_legal_text(url: str) -> dict:
    """
    Fetch official EUR-Lex legal text page and extract broad readable text.

    EUR-Lex pages can include navigation, JavaScript warnings, and complex HTML.
    This function uses broad text extraction and removes obvious non-content fragments.
    """
    headers = {
        "User-Agent": (
            "Mozilla/5.0 "
            "(compatible; ResponsibleAIRAGAssistant/1.0; educational portfolio project)"
        ),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    }

    response = requests.get(url, headers=headers, timeout=120, allow_redirects=True)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    for tag in soup(["script", "style", "noscript", "svg"]):
        tag.decompose()

    raw_lines = soup.get_text("\n", strip=True).splitlines()

    cleaned_lines = []
    skip_terms = [
        "javascript is disabled",
        "verify that you're not a robot",
        "enable javascript",
        "skip to main content",
        "site map",
        "cookies",
        "legal notice",
        "privacy statement",
        "accessibility",
        "language selection",
        "search in eur-lex",
        "document information",
        "procedure",
        "document summary",
        "share this page",
    ]

    for line in raw_lines:
        line = clean_extracted_text(line)

        if not line:
            continue

        lower_line = line.lower()

        if any(term in lower_line for term in skip_terms):
            continue

        # Keep article, recital, annex, and regulation text.
        # Remove extremely short fragments that are likely navigation.
        if len(line.split()) < 3 and not re.search(r"Article\s+\d+|ANNEX|CHAPTER|SECTION", line, re.IGNORECASE):
            continue

        cleaned_lines.append(line)

    extracted_text = "\n".join(cleaned_lines)
    cleaned_text = clean_extracted_text(extracted_text)

    return {
        "url": url,
        "status_code": response.status_code,
        "text": extracted_text,
        "cleaned_text": cleaned_text,
        "word_count": len(cleaned_text.split()),
        "character_count": len(cleaned_text),
        "preview": cleaned_text[:1000],
    }


eurlex_repair_targets = [
    {
        "source_id": "SRC-002",
        "new_raw_filename": "src_002_eurlex_ai_act_regulation_2024_1689.txt",
        "url": "https://eur-lex.europa.eu/eli/reg/2024/1689/oj/eng",
    },
    {
        "source_id": "SRC-003",
        "new_raw_filename": "src_003_eurlex_gdpr_regulation_2016_679.txt",
        "url": "https://eur-lex.europa.eu/eli/reg/2016/679/oj/eng",
    },
]

eurlex_repair_results = []

 for target in eurlex_repair_targets:
    source_id = target["source_id"]
    output_path = RAW_DATA_DIR / target["new_raw_filename"]

    print("=" * 100)
    print(f"Repairing {source_id} from official EUR-Lex text page")
    print(f"Output path: {output_path}")

    try:
        result = fetch_eurlex_legal_text(target["url"])

        output_path.write_text(result["text"], encoding="utf-8")

        raw_word_count = result["word_count"]
        raw_character_count = result["character_count"]
        raw_file_size_bytes = output_path.stat().st_size if output_path.exists() else 0
        raw_file_exists = output_path.exists()

        if raw_word_count >= 1000:
            collection_status = "collected"
            processing_status = "raw_text_saved"
            notes = "Repaired using official EUR-Lex legal text page."
        else:
            collection_status = "collection_warning"
            processing_status = "low_text_extraction"
            notes = "EUR-Lex text extraction is shorter than expected. Manual review needed."

        print(f"Status code: {result['status_code']}")
        print(f"Word count: {raw_word_count}")
        print(f"Character count: {raw_character_count}")
        print(f"File size: {raw_file_size_bytes}")
        print("Preview:")
        print(result["preview"])

        # Update source inventory
        source_inventory_df.loc[
            source_inventory_df["source_id"] == source_id,
            "url"
        ] = target["url"]

        source_inventory_df.loc[
            source_inventory_df["source_id"] == source_id,
            "planned_raw_filename"
        ] = target["new_raw_filename"]

        source_inventory_df.loc[
            source_inventory_df["source_id"] == source_id,
            "planned_raw_path"
        ] = str(output_path)

        source_inventory_df.loc[
            source_inventory_df["source_id"] == source_id,
            "collection_status"
        ] = collection_status

        source_inventory_df.loc[
            source_inventory_df["source_id"] == source_id,
            "processing_status"
        ] = processing_status

        source_inventory_df.loc[
            source_inventory_df["source_id"] == source_id,
            "raw_file_exists"
        ] = raw_file_exists

        source_inventory_df.loc[
            source_inventory_df["source_id"] == source_id,
            "raw_word_count"
        ] = raw_word_count

        source_inventory_df.loc[
            source_inventory_df["source_id"] == source_id,
            "raw_character_count"
        ] = raw_character_count

        source_inventory_df.loc[
            source_inventory_df["source_id"] == source_id,
            "raw_file_size_bytes"
        ] = raw_file_size_bytes

        source_inventory_df.loc[
            source_inventory_df["source_id"] == source_id,
            "collection_notes"
        ] = notes

    except Exception as error:
        collection_status = "collection_failed"
        processing_status = "not_processed_yet"
        raw_file_exists = output_path.exists()
        raw_word_count = np.nan
        raw_character_count = np.nan
        raw_file_size_bytes = output_path.stat().st_size if output_path.exists() else np.nan
        notes = f"EUR-Lex text repair failed: {error}"

        print(f"ERROR repairing {source_id}: {error}")

    eurlex_repair_results.append(
        {
            "source_id": source_id,
            "collection_status": collection_status,
            "processing_status": processing_status,
            "raw_file_exists": raw_file_exists,
            "raw_word_count": raw_word_count,
            "raw_character_count": raw_character_count,
            "raw_file_size_bytes": raw_file_size_bytes,
            "collection_notes": notes,
        }
    )

eurlex_repair_results_df = pd.DataFrame(eurlex_repair_results)

print("=" * 100)
print("EUR-Lex text repair run completed.")

eurlex_repair_results_df

IndentationError: unexpected indent (2380555523.py, line 92)

In [12]:
import requests
from bs4 import BeautifulSoup
import numpy as np
import pandas as pd
import re


def fetch_eurlex_legal_text(url: str) -> dict:
    """
    Fetch official EUR-Lex legal text page and extract broad readable text.

    EUR-Lex pages can include navigation, JavaScript warnings, and complex HTML.
    This function uses broad text extraction and removes obvious non-content fragments.
    """
    headers = {
        "User-Agent": (
            "Mozilla/5.0 "
            "(compatible; ResponsibleAIRAGAssistant/1.0; educational portfolio project)"
        ),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    }

    response = requests.get(url, headers=headers, timeout=120, allow_redirects=True)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    for tag in soup(["script", "style", "noscript", "svg"]):
        tag.decompose()

    raw_lines = soup.get_text("\n", strip=True).splitlines()

    cleaned_lines = []
    skip_terms = [
        "javascript is disabled",
        "verify that you're not a robot",
        "enable javascript",
        "skip to main content",
        "site map",
        "cookies",
        "legal notice",
        "privacy statement",
        "accessibility",
        "language selection",
        "search in eur-lex",
        "document information",
        "procedure",
        "document summary",
        "share this page",
    ]

    for line in raw_lines:
        line = clean_extracted_text(line)

        if not line:
            continue

        lower_line = line.lower()

        if any(term in lower_line for term in skip_terms):
            continue

        if len(line.split()) < 3 and not re.search(
            r"Article\s+\d+|ANNEX|CHAPTER|SECTION",
            line,
            re.IGNORECASE,
        ):
            continue

        cleaned_lines.append(line)

    extracted_text = "\n".join(cleaned_lines)
    cleaned_text = clean_extracted_text(extracted_text)

    return {
        "url": url,
        "status_code": response.status_code,
        "text": extracted_text,
        "cleaned_text": cleaned_text,
        "word_count": len(cleaned_text.split()),
        "character_count": len(cleaned_text),
        "preview": cleaned_text[:1000],
    }


eurlex_repair_targets = [
    {
        "source_id": "SRC-002",
        "new_raw_filename": "src_002_eurlex_ai_act_regulation_2024_1689.txt",
        "url": "https://eur-lex.europa.eu/eli/reg/2024/1689/oj/eng",
    },
    {
        "source_id": "SRC-003",
        "new_raw_filename": "src_003_eurlex_gdpr_regulation_2016_679.txt",
        "url": "https://eur-lex.europa.eu/eli/reg/2016/679/oj/eng",
    },
]

eurlex_repair_results = []

for target in eurlex_repair_targets:
    source_id = target["source_id"]
    output_path = RAW_DATA_DIR / target["new_raw_filename"]

    print("=" * 100)
    print(f"Repairing {source_id} from official EUR-Lex text page")
    print(f"Output path: {output_path}")

    try:
        result = fetch_eurlex_legal_text(target["url"])

        output_path.write_text(result["text"], encoding="utf-8")

        raw_word_count = result["word_count"]
        raw_character_count = result["character_count"]
        raw_file_size_bytes = output_path.stat().st_size if output_path.exists() else 0
        raw_file_exists = output_path.exists()

        if raw_word_count >= 1000:
            collection_status = "collected"
            processing_status = "raw_text_saved"
            notes = "Repaired using official EUR-Lex legal text page."
        else:
            collection_status = "collection_warning"
            processing_status = "low_text_extraction"
            notes = "EUR-Lex text extraction is shorter than expected. Manual review needed."

        print(f"Status code: {result['status_code']}")
        print(f"Word count: {raw_word_count}")
        print(f"Character count: {raw_character_count}")
        print(f"File size: {raw_file_size_bytes}")
        print("Preview:")
        print(result["preview"])

        source_inventory_df.loc[
            source_inventory_df["source_id"] == source_id,
            "url"
        ] = target["url"]

        source_inventory_df.loc[
            source_inventory_df["source_id"] == source_id,
            "planned_raw_filename"
        ] = target["new_raw_filename"]

        source_inventory_df.loc[
            source_inventory_df["source_id"] == source_id,
            "planned_raw_path"
        ] = str(output_path)

        source_inventory_df.loc[
            source_inventory_df["source_id"] == source_id,
            "collection_status"
        ] = collection_status

        source_inventory_df.loc[
            source_inventory_df["source_id"] == source_id,
            "processing_status"
        ] = processing_status

        source_inventory_df.loc[
            source_inventory_df["source_id"] == source_id,
            "raw_file_exists"
        ] = raw_file_exists

        source_inventory_df.loc[
            source_inventory_df["source_id"] == source_id,
            "raw_word_count"
        ] = raw_word_count

        source_inventory_df.loc[
            source_inventory_df["source_id"] == source_id,
            "raw_character_count"
        ] = raw_character_count

        source_inventory_df.loc[
            source_inventory_df["source_id"] == source_id,
            "raw_file_size_bytes"
        ] = raw_file_size_bytes

        source_inventory_df.loc[
            source_inventory_df["source_id"] == source_id,
            "collection_notes"
        ] = notes

    except Exception as error:
        collection_status = "collection_failed"
        processing_status = "not_processed_yet"
        raw_file_exists = output_path.exists()
        raw_word_count = np.nan
        raw_character_count = np.nan
        raw_file_size_bytes = output_path.stat().st_size if output_path.exists() else np.nan
        notes = f"EUR-Lex text repair failed: {error}"

        print(f"ERROR repairing {source_id}: {error}")

    eurlex_repair_results.append(
        {
            "source_id": source_id,
            "collection_status": collection_status,
            "processing_status": processing_status,
            "raw_file_exists": raw_file_exists,
            "raw_word_count": raw_word_count,
            "raw_character_count": raw_character_count,
            "raw_file_size_bytes": raw_file_size_bytes,
            "collection_notes": notes,
        }
    )

eurlex_repair_results_df = pd.DataFrame(eurlex_repair_results)

print("=" * 100)
print("EUR-Lex text repair run completed.")

eurlex_repair_results_df

Repairing SRC-002 from official EUR-Lex text page
Output path: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\raw\src_002_eurlex_ai_act_regulation_2024_1689.txt
Status code: 202
Word count: 0
Character count: 0
File size: 0
Preview:

Repairing SRC-003 from official EUR-Lex text page
Output path: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\raw\src_003_eurlex_gdpr_regulation_2016_679.txt
Status code: 202
Word count: 0
Character count: 0
File size: 0
Preview:

EUR-Lex text repair run completed.


,source_id,collection_status,processing_status,raw_file_exists,raw_word_count,raw_character_count,raw_file_size_bytes,collection_notes
0,SRC-002,collection_warning,low_text_extraction,True,0,0,0,EUR-Lex text extraction is shorter than expect...
1,SRC-003,collection_warning,low_text_extraction,True,0,0,0,EUR-Lex text extraction is shorter than expect...


In [13]:
# Validate manually downloaded EUR-Lex PDF files

manual_pdf_files = [
    {
        "source_id": "SRC-002",
        "filename": "src_002_eurlex_ai_act_regulation_2024_1689.pdf",
    },
    {
        "source_id": "SRC-003",
        "filename": "src_003_eurlex_gdpr_regulation_2016_679.pdf",
    },
]

manual_pdf_validation_results = []

for item in manual_pdf_files:
    source_id = item["source_id"]
    raw_path = RAW_DATA_DIR / item["filename"]
    
    file_exists = raw_path.exists()
    file_size_bytes = raw_path.stat().st_size if file_exists else 0
    
    if file_exists and file_size_bytes > 50_000:
        first_four_bytes = raw_path.read_bytes()[:4]
        is_valid_pdf = first_four_bytes == b"%PDF"
    else:
        is_valid_pdf = False
    
    if is_valid_pdf:
        collection_status = "collected"
        processing_status = "raw_pdf_saved"
        notes = "Manually downloaded from official EUR-Lex page after programmatic extraction failed."
    else:
        collection_status = "collection_warning"
        processing_status = "manual_pdf_validation_failed"
        notes = "Manual PDF file missing, too small, or not a valid PDF."
    
    source_inventory_df.loc[
        source_inventory_df["source_id"] == source_id,
        "planned_raw_filename"
    ] = item["filename"]
    
    source_inventory_df.loc[
        source_inventory_df["source_id"] == source_id,
        "planned_raw_path"
    ] = str(raw_path)
    
    source_inventory_df.loc[
        source_inventory_df["source_id"] == source_id,
        "collection_status"
    ] = collection_status
    
    source_inventory_df.loc[
        source_inventory_df["source_id"] == source_id,
        "processing_status"
    ] = processing_status
    
    source_inventory_df.loc[
        source_inventory_df["source_id"] == source_id,
        "raw_file_exists"
    ] = file_exists
    
    source_inventory_df.loc[
        source_inventory_df["source_id"] == source_id,
        "raw_file_size_bytes"
    ] = file_size_bytes
    
    source_inventory_df.loc[
        source_inventory_df["source_id"] == source_id,
        "collection_notes"
    ] = notes
    
    manual_pdf_validation_results.append(
        {
            "source_id": source_id,
            "filename": item["filename"],
            "file_exists": file_exists,
            "file_size_bytes": file_size_bytes,
            "is_valid_pdf": is_valid_pdf,
            "collection_status": collection_status,
            "processing_status": processing_status,
            "notes": notes,
        }
    )

manual_pdf_validation_df = pd.DataFrame(manual_pdf_validation_results)
manual_pdf_validation_df

,source_id,filename,file_exists,file_size_bytes,is_valid_pdf,collection_status,processing_status,notes
0,SRC-002,src_002_eurlex_ai_act_regulation_2024_1689.pdf,True,2583319,True,collected,raw_pdf_saved,Manually downloaded from official EUR-Lex page...
1,SRC-003,src_003_eurlex_gdpr_regulation_2016_679.pdf,True,982296,True,collected,raw_pdf_saved,Manually downloaded from official EUR-Lex page...


In [14]:
# Save source inventory after manual EUR-Lex PDF validation

source_inventory_df.to_csv(source_inventory_path, index=False, encoding="utf-8")

print(f"Saved source inventory to: {source_inventory_path}")
print(f"File exists: {source_inventory_path.exists()}")

source_inventory_df[
    [
        "source_id",
        "source_name",
        "planned_raw_filename",
        "collection_status",
        "processing_status",
        "raw_file_exists",
        "raw_file_size_bytes",
        "collection_notes",
    ]
]

Saved source inventory to: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\processed\source_inventory.csv
File exists: True


,source_id,source_name,planned_raw_filename,collection_status,processing_status,raw_file_exists,raw_file_size_bytes,collection_notes
0,SRC-001,AI Act,src_001_european_commission_ai_act_overview.txt,collected,chunked,True,NaN,
1,SRC-002,Regulation (EU) 2024/1689 Artificial Intellige...,src_002_eurlex_ai_act_regulation_2024_1689.pdf,collected,raw_pdf_saved,True,2583319.0,Manually downloaded from official EUR-Lex page...
2,SRC-003,Regulation (EU) 2016/679 General Data Protecti...,src_003_eurlex_gdpr_regulation_2016_679.pdf,collected,raw_pdf_saved,True,982296.0,Manually downloaded from official EUR-Lex page...
3,SRC-004,Artificial Intelligence Risk Management Framew...,src_004_nist_ai_rmf_1_0.pdf,collected,raw_pdf_saved,True,1946127.0,PDF downloaded successfully.
4,SRC-005,AI Risk Management Framework overview page,src_005_nist_ai_rmf_overview.txt,collected,raw_text_saved,True,3647.0,Repaired using broader text extraction.


In [15]:
# Final raw source file check after manual repair

raw_file_records = []

for _, row in source_inventory_df.iterrows():
    raw_path = RAW_DATA_DIR / row["planned_raw_filename"]

    raw_file_records.append(
        {
            "source_id": row["source_id"],
            "source_name": row["source_name"],
            "planned_raw_filename": row["planned_raw_filename"],
            "raw_file_exists": raw_path.exists(),
            "file_size_bytes": raw_path.stat().st_size if raw_path.exists() else None,
            "collection_status": row["collection_status"],
            "processing_status": row["processing_status"],
        }
    )

raw_files_status_df = pd.DataFrame(raw_file_records)
raw_files_status_df

,source_id,source_name,planned_raw_filename,raw_file_exists,file_size_bytes,collection_status,processing_status
0,SRC-001,AI Act,src_001_european_commission_ai_act_overview.txt,True,14612,collected,chunked
1,SRC-002,Regulation (EU) 2024/1689 Artificial Intellige...,src_002_eurlex_ai_act_regulation_2024_1689.pdf,True,2583319,collected,raw_pdf_saved
2,SRC-003,Regulation (EU) 2016/679 General Data Protecti...,src_003_eurlex_gdpr_regulation_2016_679.pdf,True,982296,collected,raw_pdf_saved
3,SRC-004,Artificial Intelligence Risk Management Framew...,src_004_nist_ai_rmf_1_0.pdf,True,1946127,collected,raw_pdf_saved
4,SRC-005,AI Risk Management Framework overview page,src_005_nist_ai_rmf_overview.txt,True,3647,collected,raw_text_saved


## Extract Text from All Collected Sources

All planned raw sources are now available locally.

The next step is to extract readable text from each source file:

- `.txt` files are loaded directly.
- `.pdf` files are parsed page by page.
- Extracted text is cleaned and saved for later chunking.
- A source-level extraction summary is created for quality control.

This step is important because the RAG assistant should only use source text that has been successfully extracted and validated.

In [16]:
from pypdf import PdfReader


def extract_text_from_txt(file_path: Path) -> dict:
    """
    Load text from a local .txt file.
    """
    text = file_path.read_text(encoding="utf-8", errors="ignore")
    cleaned_text = clean_extracted_text(text)

    return {
        "file_type": ".txt",
        "text": text,
        "cleaned_text": cleaned_text,
        "page_count": None,
        "word_count": len(cleaned_text.split()),
        "character_count": len(cleaned_text),
    }


def extract_text_from_pdf(file_path: Path) -> dict:
    """
    Extract text from a local PDF file page by page.
    """
    reader = PdfReader(str(file_path))

    page_texts = []

    for page_index, page in enumerate(reader.pages, start=1):
        try:
            page_text = page.extract_text() or ""
        except Exception:
            page_text = ""

        page_text = page_text.strip()

        if page_text:
            page_texts.append(
                {
                    "page_number": page_index,
                    "text": page_text,
                }
            )

    combined_text = "\n\n".join(
        f"[Page {item['page_number']}]\n{item['text']}"
        for item in page_texts
    )

    cleaned_text = clean_extracted_text(combined_text)

    return {
        "file_type": ".pdf",
        "text": combined_text,
        "cleaned_text": cleaned_text,
        "page_count": len(reader.pages),
        "pages_with_text": len(page_texts),
        "word_count": len(cleaned_text.split()),
        "character_count": len(cleaned_text),
    }


def extract_text_from_raw_file(file_path: Path) -> dict:
    """
    Route extraction based on file extension.
    """
    suffix = file_path.suffix.lower()

    if suffix == ".txt":
        return extract_text_from_txt(file_path)

    if suffix == ".pdf":
        return extract_text_from_pdf(file_path)

    raise ValueError(f"Unsupported file type: {suffix}")


print("Text extraction helper functions are ready.")

Text extraction helper functions are ready.


In [17]:
# Extract text from all collected raw source files

extracted_source_records = []
extraction_summary_records = []

for _, row in source_inventory_df.iterrows():
    source_id = row["source_id"]
    source_name = row["source_name"]
    publisher = row["publisher"]
    source_type = row["source_type"]
    raw_filename = row["planned_raw_filename"]

    raw_path = RAW_DATA_DIR / raw_filename

    print("=" * 100)
    print(f"Extracting {source_id}: {source_name}")
    print(f"Raw file: {raw_path}")

    if not raw_path.exists():
        print("Raw file missing. Skipping.")

        extraction_summary_records.append(
            {
                "source_id": source_id,
                "source_name": source_name,
                "raw_filename": raw_filename,
                "raw_file_exists": False,
                "extraction_status": "raw_file_missing",
                "file_type": None,
                "page_count": None,
                "pages_with_text": None,
                "word_count": None,
                "character_count": None,
                "processed_text_file": None,
            }
        )

        continue

    try:
        extraction_result = extract_text_from_raw_file(raw_path)

        extracted_text = extraction_result["text"]
        cleaned_text = extraction_result["cleaned_text"]

        processed_text_filename = f"{source_id.lower()}_extracted_text.txt"
        processed_text_path = PROCESSED_DATA_DIR / processed_text_filename

        processed_text_path.write_text(cleaned_text, encoding="utf-8")

        extraction_status = "extracted"

        print(f"File type: {extraction_result['file_type']}")
        print(f"Page count: {extraction_result.get('page_count')}")
        print(f"Pages with text: {extraction_result.get('pages_with_text')}")
        print(f"Word count: {extraction_result['word_count']}")
        print(f"Character count: {extraction_result['character_count']}")
        print(f"Saved extracted text to: {processed_text_path}")
        print("Preview:")
        print(cleaned_text[:700])

        extracted_source_records.append(
            {
                "source_id": source_id,
                "source_name": source_name,
                "publisher": publisher,
                "source_type": source_type,
                "raw_filename": raw_filename,
                "processed_text_filename": processed_text_filename,
                "processed_text_path": str(processed_text_path),
                "text": cleaned_text,
                "word_count": extraction_result["word_count"],
                "character_count": extraction_result["character_count"],
                "page_count": extraction_result.get("page_count"),
                "pages_with_text": extraction_result.get("pages_with_text"),
            }
        )

        extraction_summary_records.append(
            {
                "source_id": source_id,
                "source_name": source_name,
                "raw_filename": raw_filename,
                "raw_file_exists": True,
                "extraction_status": extraction_status,
                "file_type": extraction_result["file_type"],
                "page_count": extraction_result.get("page_count"),
                "pages_with_text": extraction_result.get("pages_with_text"),
                "word_count": extraction_result["word_count"],
                "character_count": extraction_result["character_count"],
                "processed_text_file": processed_text_filename,
            }
        )

        source_inventory_df.loc[
            source_inventory_df["source_id"] == source_id,
            "processing_status"
        ] = "text_extracted"

        source_inventory_df.loc[
            source_inventory_df["source_id"] == source_id,
            "raw_word_count"
        ] = extraction_result["word_count"]

        source_inventory_df.loc[
            source_inventory_df["source_id"] == source_id,
            "raw_character_count"
        ] = extraction_result["character_count"]

    except Exception as error:
        print(f"ERROR extracting {source_id}: {error}")

        extraction_summary_records.append(
            {
                "source_id": source_id,
                "source_name": source_name,
                "raw_filename": raw_filename,
                "raw_file_exists": True,
                "extraction_status": f"extraction_failed: {error}",
                "file_type": raw_path.suffix.lower(),
                "page_count": None,
                "pages_with_text": None,
                "word_count": None,
                "character_count": None,
                "processed_text_file": None,
            }
        )

extracted_sources_df = pd.DataFrame(extracted_source_records)
extraction_summary_df = pd.DataFrame(extraction_summary_records)

print("=" * 100)
print("Extraction completed.")
print(f"Sources extracted: {len(extracted_sources_df)}")

extraction_summary_df

Extracting SRC-001: AI Act
Raw file: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\raw\src_001_european_commission_ai_act_overview.txt
File type: .txt
Page count: None
Pages with text: None
Word count: 2229
Character count: 14464
Saved extracted text to: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\processed\src-001_extracted_text.txt
Preview:
The AI Act is the first-ever legal framework on AI, which addresses the risks of AI and positions Europe to play a leading role globally. The AI Act (Regulation (EU) 2024/1689 laying down harmonised rules on artificial intelligence) is the first-ever comprehensive legal framework on AI worldwide. The aim of the rules is to foster trustworthy AI in Europe. For any questions on the AI Act , check out the AI Act Single Information platform . The AI Act sets out a risk-based rules for AI developers and deployers regarding specific uses of AI. The AI Act is part of a wider package of policy measures to support the development of

,source_id,source_name,raw_filename,raw_file_exists,extraction_status,file_type,page_count,pages_with_text,word_count,character_count,processed_text_file
0,SRC-001,AI Act,src_001_european_commission_ai_act_overview.txt,True,extracted,.txt,NaN,NaN,2229,14464,src-001_extracted_text.txt
1,SRC-002,Regulation (EU) 2024/1689 Artificial Intellige...,src_002_eurlex_ai_act_regulation_2024_1689.pdf,True,extracted,.pdf,144.0,144.0,116054,621527,src-002_extracted_text.txt
2,SRC-003,Regulation (EU) 2016/679 General Data Protecti...,src_003_eurlex_gdpr_regulation_2016_679.pdf,True,extracted,.pdf,88.0,88.0,70015,370731,src-003_extracted_text.txt
3,SRC-004,Artificial Intelligence Risk Management Framew...,src_004_nist_ai_rmf_1_0.pdf,True,extracted,.pdf,48.0,48.0,16007,106959,src-004_extracted_text.txt
4,SRC-005,AI Risk Management Framework overview page,src_005_nist_ai_rmf_overview.txt,True,extracted,.txt,NaN,NaN,562,3645,src-005_extracted_text.txt


In [18]:
# Save extraction summary

extraction_summary_path = PROCESSED_DATA_DIR / "source_text_extraction_summary.csv"

extraction_summary_df.to_csv(extraction_summary_path, index=False, encoding="utf-8")
source_inventory_df.to_csv(source_inventory_path, index=False, encoding="utf-8")

print(f"Saved extraction summary to: {extraction_summary_path}")
print(f"Extraction summary exists: {extraction_summary_path.exists()}")
print(f"Saved updated source inventory to: {source_inventory_path}")
print(f"Source inventory exists: {source_inventory_path.exists()}")

extraction_summary_df[
    [
        "source_id",
        "source_name",
        "extraction_status",
        "file_type",
        "page_count",
        "pages_with_text",
        "word_count",
        "character_count",
        "processed_text_file",
    ]
]

Saved extraction summary to: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\processed\source_text_extraction_summary.csv
Extraction summary exists: True
Saved updated source inventory to: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\processed\source_inventory.csv
Source inventory exists: True


,source_id,source_name,extraction_status,file_type,page_count,pages_with_text,word_count,character_count,processed_text_file
0,SRC-001,AI Act,extracted,.txt,NaN,NaN,2229,14464,src-001_extracted_text.txt
1,SRC-002,Regulation (EU) 2024/1689 Artificial Intellige...,extracted,.pdf,144.0,144.0,116054,621527,src-002_extracted_text.txt
2,SRC-003,Regulation (EU) 2016/679 General Data Protecti...,extracted,.pdf,88.0,88.0,70015,370731,src-003_extracted_text.txt
3,SRC-004,Artificial Intelligence Risk Management Framew...,extracted,.pdf,48.0,48.0,16007,106959,src-004_extracted_text.txt
4,SRC-005,AI Risk Management Framework overview page,extracted,.txt,NaN,NaN,562,3645,src-005_extracted_text.txt


In [19]:
# Extraction quality-control check

qc_records = []

for _, row in extraction_summary_df.iterrows():
    word_count = row["word_count"]

    if pd.isna(word_count):
        quality_status = "failed"
    elif word_count < 100:
        quality_status = "too_short"
    elif word_count < 500:
        quality_status = "short_but_usable"
    else:
        quality_status = "usable"

    qc_records.append(
        {
            "source_id": row["source_id"],
            "source_name": row["source_name"],
            "extraction_status": row["extraction_status"],
            "word_count": word_count,
            "character_count": row["character_count"],
            "quality_status": quality_status,
        }
    )

extraction_qc_df = pd.DataFrame(qc_records)

print("Extraction QC summary:")
print(extraction_qc_df["quality_status"].value_counts(dropna=False))

extraction_qc_df

Extraction QC summary:
quality_status
usable    5
Name: count, dtype: int64


,source_id,source_name,extraction_status,word_count,character_count,quality_status
0,SRC-001,AI Act,extracted,2229,14464,usable
1,SRC-002,Regulation (EU) 2024/1689 Artificial Intellige...,extracted,116054,621527,usable
2,SRC-003,Regulation (EU) 2016/679 General Data Protecti...,extracted,70015,370731,usable
3,SRC-004,Artificial Intelligence Risk Management Framew...,extracted,16007,106959,usable
4,SRC-005,AI Risk Management Framework overview page,extracted,562,3645,usable


## Create Multi-Source Chunk Table

The extracted source texts are now converted into retrieval-ready chunks.

Chunking strategy:

- Use word-based chunks.
- Preserve source metadata for every chunk.
- Include source ID, source name, publisher, source type, URL, raw filename, and processed text filename.
- Use overlapping chunks to reduce the risk of losing important context at chunk boundaries.

This chunk table will later replace the earlier single-source vector store.

In [20]:
def chunk_text_by_words(text: str, chunk_size: int = 300, overlap: int = 60) -> list[str]:
    """
    Split text into overlapping word-based chunks.

    Parameters
    ----------
    text : str
        Input text to chunk.
    chunk_size : int
        Maximum number of words per chunk.
    overlap : int
        Number of words repeated between consecutive chunks.

    Returns
    -------
    list[str]
        List of text chunks.
    """
    if not isinstance(text, str) or not text.strip():
        return []

    words = text.split()

    if len(words) <= chunk_size:
        return [" ".join(words)]

    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk_words = words[start:end]
        chunk = " ".join(chunk_words).strip()

        if chunk:
            chunks.append(chunk)

        if end >= len(words):
            break

        start = end - overlap

    return chunks


def create_multisource_chunk_records(
    source_inventory: pd.DataFrame,
    extraction_summary: pd.DataFrame,
    processed_data_dir: Path,
    chunk_size: int = 300,
    overlap: int = 60,
) -> pd.DataFrame:
    """
    Create a multi-source chunk table from processed extracted text files.
    """
    all_chunk_records = []

    usable_sources = extraction_summary[
        extraction_summary["extraction_status"] == "extracted"
    ].copy()

    for _, extraction_row in usable_sources.iterrows():
        source_id = extraction_row["source_id"]
        processed_text_file = extraction_row["processed_text_file"]

        inventory_match = source_inventory[
            source_inventory["source_id"] == source_id
        ]

        if inventory_match.empty:
            print(f"Skipping {source_id}: no inventory metadata found.")
            continue

        inventory_row = inventory_match.iloc[0]

        processed_text_path = processed_data_dir / processed_text_file

        if not processed_text_path.exists():
            print(f"Skipping {source_id}: processed text file missing: {processed_text_path}")
            continue

        text = processed_text_path.read_text(encoding="utf-8", errors="ignore")
        text = clean_extracted_text(text)

        chunks = chunk_text_by_words(
            text=text,
            chunk_size=chunk_size,
            overlap=overlap,
        )

        print("=" * 100)
        print(f"Source: {source_id} — {inventory_row['source_name']}")
        print(f"Processed text file: {processed_text_file}")
        print(f"Source words: {len(text.split())}")
        print(f"Chunks created: {len(chunks)}")

        for chunk_index, chunk_text in enumerate(chunks, start=1):
            chunk_id = f"{source_id}_CHUNK_{chunk_index:04d}"

            all_chunk_records.append(
                {
                    "chunk_id": chunk_id,
                    "source_id": source_id,
                    "source_name": inventory_row["source_name"],
                    "publisher": inventory_row["publisher"],
                    "source_type": inventory_row["source_type"],
                    "main_topic": inventory_row["main_topic"],
                    "url": inventory_row["url"],
                    "planned_use": inventory_row["planned_use"],
                    "raw_filename": inventory_row["planned_raw_filename"],
                    "processed_text_file": processed_text_file,
                    "chunk_index": chunk_index,
                    "chunk_text": chunk_text,
                    "word_count": len(chunk_text.split()),
                    "character_count": len(chunk_text),
                    "chunk_size_setting": chunk_size,
                    "chunk_overlap_setting": overlap,
                }
            )

    chunks_df = pd.DataFrame(all_chunk_records)

    return chunks_df


print("Multi-source chunking helper functions are ready.")

Multi-source chunking helper functions are ready.


In [21]:
# Create multi-source chunk table

MULTISOURCE_CHUNK_SIZE = 300
MULTISOURCE_CHUNK_OVERLAP = 60

multisource_chunks_df = create_multisource_chunk_records(
    source_inventory=source_inventory_df,
    extraction_summary=extraction_summary_df,
    processed_data_dir=PROCESSED_DATA_DIR,
    chunk_size=MULTISOURCE_CHUNK_SIZE,
    overlap=MULTISOURCE_CHUNK_OVERLAP,
)

print("=" * 100)
print("Multi-source chunking completed.")
print(f"Total chunks created: {len(multisource_chunks_df)}")
print(f"Sources included: {multisource_chunks_df['source_id'].nunique()}")
print(f"Average words per chunk: {multisource_chunks_df['word_count'].mean():.1f}")
print(f"Minimum words per chunk: {multisource_chunks_df['word_count'].min()}")
print(f"Maximum words per chunk: {multisource_chunks_df['word_count'].max()}")

multisource_chunks_df.head()

Source: SRC-001 — AI Act
Processed text file: src-001_extracted_text.txt
Source words: 2229
Chunks created: 10
Source: SRC-002 — Regulation (EU) 2024/1689 Artificial Intelligence Act
Processed text file: src-002_extracted_text.txt
Source words: 116054
Chunks created: 484
Source: SRC-003 — Regulation (EU) 2016/679 General Data Protection Regulation
Processed text file: src-003_extracted_text.txt
Source words: 70015
Chunks created: 292
Source: SRC-004 — Artificial Intelligence Risk Management Framework AI RMF 1.0
Processed text file: src-004_extracted_text.txt
Source words: 16007
Chunks created: 67
Source: SRC-005 — AI Risk Management Framework overview page
Processed text file: src-005_extracted_text.txt
Source words: 562
Chunks created: 3
Multi-source chunking completed.
Total chunks created: 856
Sources included: 5
Average words per chunk: 299.0
Minimum words per chunk: 69
Maximum words per chunk: 300


,chunk_id,source_id,source_name,publisher,source_type,main_topic,url,planned_use,raw_filename,processed_text_file,chunk_index,chunk_text,word_count,character_count,chunk_size_setting,chunk_overlap_setting
0,SRC-001_CHUNK_0001,SRC-001,AI Act,European Commission,Official web page,EU AI Act overview and risk-based approach,https://digital-strategy.ec.europa.eu/en/polic...,Governance overview and user-facing source exp...,src_001_european_commission_ai_act_overview.txt,src-001_extracted_text.txt,1,The AI Act is the first-ever legal framework o...,300,1745,300,60
1,SRC-001_CHUNK_0002,SRC-001,AI Act,European Commission,Official web page,EU AI Act overview and risk-based approach,https://digital-strategy.ec.europa.eu/en/polic...,Governance overview and user-facing source exp...,src_001_european_commission_ai_act_overview.txt,src-001_extracted_text.txt,2,The AI Act ensures that Europeans can trust wh...,300,1970,300,60
2,SRC-001_CHUNK_0003,SRC-001,AI Act,European Commission,Official web page,EU AI Act overview and risk-based approach,https://digital-strategy.ec.europa.eu/en/polic...,Governance overview and user-facing source exp...,src_001_european_commission_ai_act_overview.txt,src-001_extracted_text.txt,3,the practical application of the prohibited pr...,300,2051,300,60
3,SRC-001_CHUNK_0004,SRC-001,AI Act,European Commission,Official web page,EU AI Act overview and risk-based approach,https://digital-strategy.ec.europa.eu/en/polic...,Governance overview and user-facing source exp...,src_001_european_commission_ai_act_overview.txt,src-001_extracted_text.txt,4,processes (e.g. AI solutions to prepare court ...,300,1896,300,60
4,SRC-001_CHUNK_0005,SRC-001,AI Act,European Commission,Official web page,EU AI Act overview and risk-based approach,https://digital-strategy.ec.europa.eu/en/polic...,Governance overview and user-facing source exp...,src_001_european_commission_ai_act_overview.txt,src-001_extracted_text.txt,5,fall into this category. This includes applica...,300,1874,300,60


In [22]:
# Multi-source chunk summary by source

chunk_summary_by_source_df = (
    multisource_chunks_df
    .groupby(["source_id", "source_name", "publisher"], as_index=False)
    .agg(
        chunk_count=("chunk_id", "count"),
        total_chunk_words=("word_count", "sum"),
        average_chunk_words=("word_count", "mean"),
        min_chunk_words=("word_count", "min"),
        max_chunk_words=("word_count", "max"),
    )
)

chunk_summary_by_source_df["average_chunk_words"] = (
    chunk_summary_by_source_df["average_chunk_words"].round(1)
)

chunk_summary_by_source_df

,source_id,source_name,publisher,chunk_count,total_chunk_words,average_chunk_words,min_chunk_words,max_chunk_words
0,SRC-001,AI Act,European Commission,10,2769,276.9,69,300
1,SRC-002,Regulation (EU) 2024/1689 Artificial Intellige...,EUR-Lex / European Union,484,145034,299.7,134,300
2,SRC-003,Regulation (EU) 2016/679 General Data Protecti...,EUR-Lex / European Union,292,87475,299.6,175,300
3,SRC-004,Artificial Intelligence Risk Management Framew...,NIST,67,19967,298.0,167,300
4,SRC-005,AI Risk Management Framework overview page,NIST,3,682,227.3,82,300


In [23]:
# Validate multi-source chunk table

expected_sources = set(source_inventory_df["source_id"])
actual_sources = set(multisource_chunks_df["source_id"])

missing_sources = expected_sources - actual_sources
unexpected_sources = actual_sources - expected_sources

empty_chunks = multisource_chunks_df[
    multisource_chunks_df["chunk_text"].isna()
    | (multisource_chunks_df["chunk_text"].str.strip() == "")
]

too_short_chunks = multisource_chunks_df[
    multisource_chunks_df["word_count"] < 30
]

validation_results = {
    "expected_source_count": len(expected_sources),
    "actual_source_count": len(actual_sources),
    "missing_sources": sorted(list(missing_sources)),
    "unexpected_sources": sorted(list(unexpected_sources)),
    "total_chunks": len(multisource_chunks_df),
    "empty_chunk_count": len(empty_chunks),
    "too_short_chunk_count": len(too_short_chunks),
    "min_words_per_chunk": int(multisource_chunks_df["word_count"].min()),
    "max_words_per_chunk": int(multisource_chunks_df["word_count"].max()),
}

for key, value in validation_results.items():
    print(f"{key}: {value}")

if missing_sources:
    print("WARNING: Some expected sources are missing from the chunk table.")
elif len(empty_chunks) > 0:
    print("WARNING: Empty chunks found.")
else:
    print("Multi-source chunk validation passed.")

expected_source_count: 5
actual_source_count: 5
missing_sources: []
unexpected_sources: []
total_chunks: 856
empty_chunk_count: 0
too_short_chunk_count: 0
min_words_per_chunk: 69
max_words_per_chunk: 300
Multi-source chunk validation passed.


In [24]:
# Save multi-source chunk table and summary

multisource_chunks_path = PROCESSED_DATA_DIR / "multisource_chunks.csv"
chunk_summary_path = PROCESSED_DATA_DIR / "multisource_chunk_summary.csv"

multisource_chunks_df.to_csv(multisource_chunks_path, index=False, encoding="utf-8")
chunk_summary_by_source_df.to_csv(chunk_summary_path, index=False, encoding="utf-8")

print(f"Saved multi-source chunks to: {multisource_chunks_path}")
print(f"Multi-source chunks file exists: {multisource_chunks_path.exists()}")
print(f"Rows saved: {len(multisource_chunks_df)}")

print("-" * 80)

print(f"Saved chunk summary to: {chunk_summary_path}")
print(f"Chunk summary file exists: {chunk_summary_path.exists()}")
print(f"Rows saved: {len(chunk_summary_by_source_df)}")

Saved multi-source chunks to: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\processed\multisource_chunks.csv
Multi-source chunks file exists: True
Rows saved: 856
--------------------------------------------------------------------------------
Saved chunk summary to: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\processed\multisource_chunk_summary.csv
Chunk summary file exists: True
Rows saved: 5


## Notebook 04 Checkpoint

Notebook 04 completed the source expansion and multi-source preparation stage.

Completed work:

- Loaded the existing source inventory.
- Collected and repaired additional authoritative public sources.
- Validated manually downloaded EUR-Lex PDF files.
- Extracted text from all collected raw sources.
- Created a multi-source chunk table.
- Saved source-level extraction summaries.
- Saved chunk-level retrieval-ready data.

The next notebook will rebuild the vector store using the complete multi-source chunk table.

In [25]:
# Final Notebook 04 checkpoint

notebook_04_checkpoint = {
    "sources_in_inventory": len(source_inventory_df),
    "sources_extracted": len(extraction_summary_df),
    "usable_sources": int((extraction_qc_df["quality_status"] == "usable").sum()),
    "total_multisource_chunks": len(multisource_chunks_df),
    "sources_in_chunk_table": multisource_chunks_df["source_id"].nunique(),
    "multisource_chunks_file_exists": multisource_chunks_path.exists(),
    "chunk_summary_file_exists": chunk_summary_path.exists(),
}

for key, value in notebook_04_checkpoint.items():
    print(f"{key}: {value}")

sources_in_inventory: 5
sources_extracted: 5
usable_sources: 5
total_multisource_chunks: 856
sources_in_chunk_table: 5
multisource_chunks_file_exists: True
chunk_summary_file_exists: True
